# Data Preparation for Machine Learning using Pandas


---
# Setup & Loading the Data

We'll use a handful of libraries today, each with a specific job:

| Library | Job |
|---|---|
| **pandas** | Loading and manipulating tabular data |
| **numpy** | Numerical operations (pandas is built on top of it) |
| **matplotlib / seaborn** | Data visualization |
| **scikit-learn** | Preprocessing tools (encoding, scaling) and the train/test split |

First upload `bigmart_sales.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Optional, if running in Colab and the file isn't uploaded yet:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv("bigmart_sales.csv")
print("Loaded! Shape:", df.shape)
df.head()

### Try it — Loading the Data

1. Print the number of rows and columns separately (not as a tuple) using `.shape`.
2. Print the list of all column names using `.columns`.

In [ ]:
# TODO: print rows and columns separately
n_rows = ____
n_cols = ____
print(f"Rows: {n_rows}, Columns: {n_cols}")

# TODO: print the column names as a list

---
# Initial Exploration

Before touching anything, get a feel for what you're working with: how many rows, what types each column holds,
and whether there's anything obviously wrong (duplicate rows, unexpected data types).

- `.info()` — column names, data types, and non-null counts in one view
- `.dtypes` — just the data types
- `.nunique()` — how many distinct values each column has (helps spot ID columns vs. categories)
- `.duplicated().sum()` — counts fully duplicated rows

In [ ]:
df.info()

In [ ]:
print("Unique values per column:")
print(df.nunique())
print()
print("Duplicate rows:", df.duplicated().sum())

### Try it — Initial Exploration

1. Print the unique values in `Outlet_Type` using `.unique()`.
2. Print the value counts (frequency of each category) for `Item_Type` using `.value_counts()`.

In [ ]:
# TODO: unique values in Outlet_Type

# TODO: value counts for Item_Type

---
# Summarizing the Data

`.describe()` gives you the standard descriptive statistics — count, mean, std, min, quartiles, max — for every
numeric column at once. This is usually the fastest way to spot something suspicious (like a minimum of `0` where
that shouldn't be possible — we'll come back to that).

In [ ]:
df.describe()

### Try it — Summarizing the Data

Using individual pandas methods (`.mean()`, `.median()`, `.std()`), print the mean, median, and standard deviation
of `Item_Outlet_Sales` — the column we'll eventually try to predict.

In [ ]:
# TODO: mean, median, std of Item_Outlet_Sales
print("Mean:", ____)
print("Median:", ____)
print("Std:", ____)

---
# Data Visualization

Numbers in a table only tell you so much — plots reveal shape, skew, and outliers at a glance. Four plot types cover
most of what you need at this stage:

- **Histogram** — the distribution of a single numeric column
- **Count plot (bar chart)** — the frequency of each category in a categorical column
- **Box plot** — median, spread, and outliers of a numeric column
- **Correlation heatmap** — how strongly numeric columns move together

In [ ]:
# Histogram: distribution of Item_MRP (item price)
plt.figure(figsize=(7, 4))
plt.hist(df['Item_MRP'], bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Item_MRP')
plt.xlabel('Item_MRP')
plt.ylabel('Count')
plt.show()

In [ ]:
# Count plot: how many products come from each Outlet_Type
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='Outlet_Type')
plt.title('Number of Items by Outlet Type')
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Box plot: spread and outliers in Item_Outlet_Sales
plt.figure(figsize=(7, 4))
sns.boxplot(x=df['Item_Outlet_Sales'], color='lightcoral')
plt.title('Spread of Item_Outlet_Sales (look at the dots past the whiskers)')
plt.show()

In [ ]:
# Correlation heatmap: numeric columns only
numeric_df = df.select_dtypes(include=['int64', 'float64'])
plt.figure(figsize=(6, 5))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Between Numeric Columns')
plt.show()

### Try it — Data Visualization

Plot a histogram of `Item_Weight` (30 bins is a good default). What shape does the distribution have — is it roughly
symmetric, or skewed to one side?

In [ ]:
# TODO: histogram of Item_Weight

---
# Handling Missing Values

Most real datasets have gaps. Two columns here do: `Item_Weight` (numeric) and `Outlet_Size` (categorical) —
and each type needs a different strategy.

| Column type | Common strategy |
|---|---|
| Numeric | Fill with the **mean** or **median** (median is safer when the data is skewed or has outliers) |
| Categorical | Fill with the **mode** (the most frequent category) |
| Either | Drop the rows/column, *only* if very little data is missing |

**From here on, every Demo cell in this notebook modifies `df` directly** — this is the real cleaning pipeline.

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

In [ ]:
# Numeric column -> fill with the median
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].median())

# Categorical column -> fill with the mode (most frequent value)
mode_outlet_size = df['Outlet_Size'].mode()[0]
df['Outlet_Size'] = df['Outlet_Size'].fillna(mode_outlet_size)

print("Missing values after cleaning:")
print(df.isnull().sum())

### Try it — Missing Values

Confirm the cleaning worked: check that `df.isnull().sum().sum()` — the *total* count of missing values across the
entire dataframe — is exactly `0`.

In [ ]:
# TODO: total missing value count across the whole dataframe
total_missing = ____
print("Total missing values:", total_missing)

---
# Cleaning Inconsistent & Disguised Data

Two extra problems that `.isnull().sum()` completely misses, because they're not technically "missing":

1. **Inconsistent category labels** — the same real-world category, spelled differently. Watch what happens
when we check `Item_Fat_Content`.
2. **Disguised missing values** — a value that *looks* valid but is actually a stand-in for "unknown," like a
visibility of exactly `0`. A product on a shelf can't realistically have *zero* visibility — that's almost
certainly missing data that got recorded as `0` instead of blank.

In [ ]:
print("Raw Item_Fat_Content categories:")
print(df['Item_Fat_Content'].value_counts())

In [ ]:
# Standardize inconsistent labels into just two real categories
fat_content_map = {
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular',
}
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)

print("Cleaned categories:")
print(df['Item_Fat_Content'].value_counts())

In [ ]:
# Disguised missing values: Item_Visibility == 0 isn't physically realistic
print("Rows with zero visibility:", (df['Item_Visibility'] == 0).sum())

# Treat 0 as missing, then fill using the average visibility for that item's Item_Type
# (a smarter fill than a single global average, since visibility norms differ by product type)
df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)
df['Item_Visibility'] = df.groupby('Item_Type')['Item_Visibility'].transform(lambda x: x.fillna(x.mean()))

print("Rows with zero/missing visibility after fix:", (df['Item_Visibility'] == 0).sum() + df['Item_Visibility'].isnull().sum())

### Try it — Inconsistent & Disguised Data

1. Confirm `Item_Fat_Content` now has exactly `2` unique values using `.nunique()`.
2. Print `df['Item_Visibility'].min()` — it should no longer be `0`.

In [ ]:
# TODO: confirm 2 unique fat content categories

# TODO: print the new minimum visibility

---
# Feature Engineering

Feature engineering means creating **new** columns from the ones you already have, when those new columns carry
information a model couldn't easily extract on its own. Two good candidates here:

- `Item_Identifier` always starts with a 2-letter code — `FD` (Food), `DR` (Drinks), or `NC` (Non-Consumable).
  That's a real category hiding inside an ID string.
- `Outlet_Establishment_Year` is a year, but what usually matters for sales is **how old** the outlet is, not
  the specific calendar year. (This dataset is understood to be from 2013, so we'll use that as our reference year.)

In [ ]:
# Extract a category from the first 2 characters of the item ID
prefix_map = {'FD': 'Food', 'DR': 'Drinks', 'NC': 'Non-Consumable'}
df['Item_Category'] = df['Item_Identifier'].str[:2].map(prefix_map)
print(df['Item_Category'].value_counts())

In [ ]:
# Convert establishment year into outlet age (reference year: 2013)
df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']
df[['Outlet_Establishment_Year', 'Outlet_Age']].drop_duplicates().sort_values('Outlet_Age')

### Try it — Feature Engineering

Create a new column `Item_MRP_Level` that bins `Item_MRP` into 4 equal-sized groups (quartiles), using
`pd.qcut(df['Item_MRP'], 4, labels=['Low', 'Medium', 'High', 'Very High'])`. Print the resulting value counts.

In [ ]:
# TODO: create the binned column
df['Item_MRP_Level'] = ____

print(df['Item_MRP_Level'].value_counts())

---
# Outlier Detection

An **outlier** is a value far outside the normal range for that column. The most common rule of thumb is the
**IQR method**: anything below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR` (where IQR = Q3 - Q1) is flagged.

Outliers aren't automatically wrong — sometimes they're genuine, important data points (a real bestseller item,
for instance). The right response is usually to **investigate first, and only remove or cap them if you have good
reason to** — which is why we'll detect them here without automatically deleting anything.

In [ ]:
Q1 = df['Item_Visibility'].quantile(0.25)
Q3 = df['Item_Visibility'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Item_Visibility'] < lower_bound) | (df['Item_Visibility'] > upper_bound)]
print(f"Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"Number of outlier rows in Item_Visibility: {len(outliers)}")

### Try it — Outlier Detection

Using the same IQR method, compute the lower and upper bounds for `Item_Outlet_Sales` (our target column),
and count how many rows fall outside them.

In [ ]:
# TODO: IQR bounds for Item_Outlet_Sales
Q1_sales = ____
Q3_sales = ____
IQR_sales = ____
lower_sales = ____
upper_sales = ____

sales_outliers = df[(df['Item_Outlet_Sales'] < lower_sales) | (df['Item_Outlet_Sales'] > upper_sales)]
print("Number of outlier rows in Item_Outlet_Sales:", len(sales_outliers))

---
# Encoding Categorical Data

ML models need numbers, not text — so every categorical column has to be converted before modeling. There are two
main approaches, and picking the right one matters:

| Type | Example column | Technique |
|---|---|---|
| **Ordinal** (categories have a natural order) | `Outlet_Size`: Small < Medium < High | Map to integers by hand, preserving the order |
| **Nominal** (no natural order) | `Outlet_Type`, `Item_Fat_Content`, `Item_Category` | **One-hot encoding** — a new 0/1 column per category |

Using plain integers (0, 1, 2, 3...) for a *nominal* column would trick the model into thinking one category is
"greater than" another, which isn't true — that's why nominal columns get one-hot encoded instead.

In [ ]:
# Ordinal encoding: Outlet_Size has a natural small-to-large order
size_map = {'Small': 0, 'Medium': 1, 'High': 2}
df['Outlet_Size_Encoded'] = df['Outlet_Size'].map(size_map)
df[['Outlet_Size', 'Outlet_Size_Encoded']].drop_duplicates()

In [ ]:
# One-hot encoding: nominal columns -> pd.get_dummies()
nominal_cols = ['Item_Fat_Content', 'Outlet_Type', 'Outlet_Location_Type', 'Item_Category']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# drop_first=True avoids redundant columns (e.g. if Fat_Content isn't 'Regular', it must be 'Low Fat')
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print("New one-hot columns created:", new_cols)

### Try it — Encoding

One-hot encode `Item_Type` on its own (16 categories) into a separate variable `item_type_dummies` using
`pd.get_dummies(df['Item_Type'], prefix='ItemType')`, and print its shape. This is a good example of a
**high-cardinality** categorical column — one with many categories — where one-hot encoding creates a lot of new columns;
in practice this is a trade-off worth thinking about rather than doing automatically.

In [ ]:
# TODO: one-hot encode Item_Type into its own variable
item_type_dummies = ____

print("Shape:", item_type_dummies.shape)

---
# Feature Scaling

`Item_MRP` ranges into the hundreds, while `Item_Visibility` is a small fraction — many ML algorithms are sensitive
to features being on wildly different scales, and will implicitly treat the bigger-magnitude column as "more important"
even when it isn't. **Scaling** puts every numeric column on a comparable range.

| Scaler | What it does | When to use |
|---|---|---|
| `StandardScaler` | Rescales to mean `0`, std `1` | The default choice for most models |
| `MinMaxScaler` | Rescales to a fixed range, usually `[0, 1]` | When you need bounded values, e.g. for neural networks |

In [ ]:
numeric_cols = ['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Age']

scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

df_encoded[numeric_cols].describe()

### Try it — Feature Scaling

Using a **fresh copy** of the original numeric columns (so we don't disturb the pipeline), apply `MinMaxScaler`
instead and compare the resulting min/max to StandardScaler's mean/std above.

In [ ]:
numeric_cols = ['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Age']
df_minmax = df[numeric_cols].copy()

# TODO: apply MinMaxScaler
minmax_scaler = ____
df_minmax[numeric_cols] = ____

df_minmax.describe()

---
# Train-Test Split

Remember step 4 of the ML workflow from Day 1: before training any model, hold out a portion of the data it will
never see, so you can honestly evaluate it afterward. `train_test_split` does this with one line — and shuffles the
data first, so the split isn't biased by row order.

In [ ]:
# Build the final modeling table: drop ID/text columns that aren't useful as model inputs
drop_cols = ['Item_Identifier', 'Outlet_Identifier', 'Item_Type', 'Outlet_Size',
             'Outlet_Establishment_Year']
df_model = df_encoded.drop(columns=drop_cols)

X = df_model.drop(columns=['Item_Outlet_Sales'])
y = df_model['Item_Outlet_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

### Try it — Train-Test Split

Redo the split with `test_size=0.3` instead of `0.2` (keep `random_state=42`), storing the results in new variables
`X_train30`, `X_test30`, `y_train30`, `y_test30`. Print all four shapes — how much bigger is the test set now?

In [ ]:
# TODO: split with test_size=0.3
X_train30, X_test30, y_train30, y_test30 = ____

print(X_train30.shape, X_test30.shape, y_train30.shape, y_test30.shape)

---
# Saving the Prepared Data & Wrap-up

The whole point of today's work is to hand off a clean, model-ready dataset — so let's save it.

In [ ]:
df_model.to_csv("bigmart_sales_cleaned.csv", index=False)
print("Saved bigmart_sales_cleaned.csv —", df_model.shape)